# Professional Practice X2: Dimensionality Reduction Pipeline

Build an end-to-end pipeline comparing PCA, t-SNE, and UMAP.

**Goal:** Compare dimensionality reduction methods on high-dimensional data.

**Pipeline:**
1. Load high-dimensional dataset
2. Apply PCA, t-SNE, UMAP
3. Visualize results
4. Evaluate preservation of structure

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
print("✅ Loaded!")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler

# Load high-dimensional data
digits = load_digits()
X, y = digits.data, digits.target

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply dimensionality reduction
print("Applying dimensionality reduction methods...")

# PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.3f}")

# t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_scaled)
print("t-SNE complete")

# UMAP
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15)
X_umap = reducer.fit_transform(X_scaled)
print("UMAP complete")

# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

methods = [
    ('PCA', X_pca),
    ('t-SNE', X_tsne),
    ('UMAP', X_umap)
]

for ax, (name, X_reduced) in zip(axes, methods):
    scatter = ax.scatter(X_reduced[:, 0], X_reduced[:, 1], 
                        c=y, cmap='tab10', s=20, alpha=0.7)
    ax.set_title(f'{name} (64D → 2D)', fontweight='bold', fontsize=14)
    ax.set_xlabel('Component 1')
    ax.set_ylabel('Component 2')
    
plt.colorbar(scatter, ax=axes, label='Digit Class')
plt.suptitle('Dimensionality Reduction Comparison on Digits Dataset', 
             fontweight='bold', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# Quantitative comparison: k-NN preservation
from sklearn.neighbors import NearestNeighbors

def neighbor_preservation(X_high, X_low, k=10):
    """Compute how well k-nearest neighbors are preserved"""
    nn_high = NearestNeighbors(n_neighbors=k+1).fit(X_high)
    nn_low = NearestNeighbors(n_neighbors=k+1).fit(X_low)
    
    neighbors_high = nn_high.kneighbors(X_high, return_distance=False)[:, 1:]
    neighbors_low = nn_low.kneighbors(X_low, return_distance=False)[:, 1:]
    
    preservation = []
    for i in range(len(X_high)):
        intersection = len(set(neighbors_high[i]) & set(neighbors_low[i]))
        preservation.append(intersection / k)
    
    return np.mean(preservation)

print("\nNeighbor Preservation (k=10):")
print("="*40)
for name, X_reduced in methods:
    preservation = neighbor_preservation(X_scaled, X_reduced, k=10)
    print(f"{name:10s}: {preservation:.3f}")

print("\n✅ Dimensionality reduction pipeline complete!")